In [ ]:
# pandas
import pandas as pd 

In [ ]:
import PySimpleGUI as sg
def selection(option):
    #sg.theme('DarkAmber')   # Add a touch of color
    #select the variable we want to calculate
    options = option

    # All the stuff inside your window.
    layout = [ 
            [sg.Text('Select one->'), sg.Listbox(options,select_mode=sg.LISTBOX_SELECT_MODE_SINGLE,size=(20,len(options)))],
            [sg.Button('Ok'), sg.Button('Cancel')]
        ]

    # Create the Window
    window = sg.Window('Make your choice', layout)

    # Event Loop to process "events" and get the "values" of the input
    while True:
        event, values = window.read()
        print( f"event={event}" )
        if event is None or event == 'Ok' or event == 'Cancel': # if user closes window or clicks cancel
            break
        
    # close  the window        
    window.close()

    if event == "Cancel":
        print( "You cancelled" )
    else:
        print('You entered ', values[0])
        sg.popup( f"You selected {values[0]}" )        
    return values


In [ ]:
#nama_10_a64 National accounts aggregates by industry (up to NACE A*64)
#NAMA_10_A64_P5 Gross capital formation by industry (up to NACE A*64)
#National accounts employment data by industry (up to NACE A*64)

import eurostat
df = eurostat.get_data_df('nama_10_nfa_st', flags=False)
df

In [ ]:
df.columns

In [ ]:
#eliminate columns for years previous to 2011
years =[]
year_start=1975
year_study=2011
for y in range(year_start,year_study): 
    years.append(y)
years.append(2020)
df=df.drop(years, axis=1)#quitamos todas las columnas de a?osque no nos interesan
#rename geo time column to be easier to handle because of the name
df.rename(columns={"geo\\time": "geo"}, inplace=True)
#We can get the unique values of the different variables

In [ ]:
asset= df.asset10.unique()
asset

N111G -Dwellings (gross)
N111N -Dwellings (net)
N112G -Other buildings and structures (gross)
N112N -Other buildings and structures (net)
N1131G -Transport equipment (gross)
N1131N -Transport equipment (net)
N11321G -Computer hardware (gross)
N11321N -Computer hardware (net)
N11322G -Telecommunications equipment (gross)
N11322N -Telecommunications equipment (net)
N1132G -ICT equipment (gross)
N1132N -ICT equipment (net)
N115G -Cultivated biological resources (gross)
N115N -Cultivated biological resources (net)
N1171G -Research and development (gross)
N1171N -Research and development (net)
N1173G -Computer software and databases (gross)
N1173N -Computer software and databases (net)
N117G -Intellectual property products (gross)
N117N -Intellectual property products (net)
N11G -Total fixed assets (gross)
N11KG -Total Construction (gross)
N11KN -Total Construction (net)
N11MG -Machinery and equipment and weapons systems (gross)
N11MN -Machinery and equipment and weapons systems (net)
N11N -Total fixed assets (net)
N11OG -Other machinery and equipment and weapons systems (gross)
N11ON -Other machinery and equipment and weapons systems (net)

In [ ]:
dic_asset = eurostat.get_dic('asset10') #dictionary for all the unit in eurostat
dic_asset

In [ ]:
#print the units in the database
assets=[]
for u in df.asset10.unique():
    if u in dic_asset:
        assets.append(dic_asset[u])
#select the unit that we want
assets

In [ ]:
df.geo.unique()

In [ ]:
df.unit.unique()

CLV15_MEUR 	Chain linked volumes (2015), million euro
	CLV15_MNAC 	Chain linked volumes (2015), million units of n...
	CRC_MEUR 	Current replacement costs, million euro
	CRC_MNAC 	Current replacement costs, million units of nat...
	PYR_MEUR 	Previous year replacement costs, million euro
	PYR_MNAC 	Previous year replacement costs, million units ... 

In [ ]:
unit=["CLV15_MEUR"]

In [ ]:
cols = ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI','FR', 'HR', 'HU', 'IE', 'IT', 'LT','LU', 'LV', 'MT','NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']
countries= set(cols)

In [ ]:
df.nace_r2.unique()

In [ ]:
weights_target = pd.read_excel("ecosystemweights.xlsx")
weights_target.NACE_R2.unique()

In [ ]:
nace_equal=set(weights_target.NACE_R2.unique())&set(df.nace_r2.unique())
nace_equal

In [ ]:
nace_dif=set(weights_target.NACE_R2.unique())-set(df.nace_r2.unique())
nace_dif

In [ ]:
#ratios to divide N80_82 and 31_32. Value Added from SBS
ratios_values = pd.read_excel("ratioCN-Value added at factor cost - million euro.xlsx")
ratios_values.set_index('nace_r2', inplace=True)#nace como ?ndice
ratios_values

In [ ]:
nace_include=['C31_C32','N80-N82']
nace_sel=set(nace_equal)|set(nace_include)
nace_sel

In [ ]:
#we select the rows that have the right unit, nace codes, items and countries
df_selection=df[df.asset10.isin(asset)&df.nace_r2.isin(nace_sel)&df.geo.isin(countries)&df.unit.isin(unit)]
df_selection

In [ ]:
import numpy as np


for z in range(0,len(asset)):
    #create a dataframe for the results
    results_compiled = pd.DataFrame()
    #print the variable that we are going to calculate
    print(asset[z], assets[z])
    var=[asset[z]]#to pass it to a list to be able to apply isin
    #get the name of the variable and name it as the variable of interest
    variable_of_interest=asset[z]
    df_selection=df[df.asset10.isin(var)&df.nace_r2.isin(nace_sel)&df.geo.isin(countries)&df.unit.isin(unit)]
    if set(df_selection.geo.unique())==set(countries):
        print("No missing countries in the data") 
        #loop to calculate values for years
        for j in range (2011,2020):
            year=j
            #we transpose the values for the year selected
            for i in range (year,year+1):
                df_target = pd.pivot_table(df_selection, values=i, index=['nace_r2'],columns=["geo"], aggfunc=np.sum, fill_value=0)#reorde
            df_target['sum'] = df_target[cols].sum(axis=1)#sums the columns of the countries to "sum" 
            moddf_target = df_target.rename({'sum': 'EU27_2020'}, axis=1)  # rename column
            #add rows C31,... and assign value with the ratio multiplying by the value of position C31_32
            year_ratio=j-2011 
            moddf_target.loc['C31'] = ratios_values.iloc[0][year_ratio]*moddf_target.loc['C31_C32','EU27_2020']
            moddf_target.loc['C32'] = ratios_values.iloc[1][year_ratio]*moddf_target.loc['C31_C32','EU27_2020']
            moddf_target.loc['N80'] = ratios_values.iloc[2][year_ratio]*moddf_target.loc['N80-N82','EU27_2020']
            moddf_target.loc['N81'] = ratios_values.iloc[3][year_ratio]*moddf_target.loc['N80-N82','EU27_2020']
            moddf_target.loc['N82'] = ratios_values.iloc[4][year_ratio]*moddf_target.loc['N80-N82','EU27_2020']
            #we eliminate rows C31_C32
            moddf_target = moddf_target.drop(['C31_C32', 'N80-N82'])
            pd.set_option('display.float_format', lambda x: '%.4f' % x)
            #cargamos los pesos en el dataframe weight  
            weights_target = pd.read_excel("ecosystemweights.xlsx")
            weights_target.set_index('NACE_R2', inplace=True)#nace como ?ndice
            column_name=str(variable_of_interest)+str(year)
            weights_target[variable_of_interest]=weights_target.index.map(moddf_target['EU27_2020'])#busca en df_target el nace y copia el valor de EU27_2020 en la fila del nace the weights_target
            #creamos la matriz de resultados
            results=weights_target
            for i in range(0,14):
                results.iloc[:, i]=weights_target.iloc[:, i]*weights_target[variable_of_interest]#we multiply the total value added by the weight for each nace
            results.loc['total'] = results.select_dtypes(pd.np.number).sum() #we add the values of each colum in a row call total
            results_compiled[column_name]=results.loc['total']
        results_compiled = results_compiled.drop(variable_of_interest)
        #results_compiled.loc['total'] = results_compiled.select_dtypes(pd.np.number).sum()
        display(results_compiled) 
        #save to file
        filename = "%s.xlsx" % variable_of_interest
        results_compiled.to_excel(filename)
    else:
        print("Missing countries in the data:", set(countries)-set(df_selection.geo.unique()))